# 🎙 Vlastní hlas pro Překladač (Piper) – trénování zdarma v Google Colab

Z vašich nahrávek (ZIP z **Překladače → Nastavení → Součásti → Vlastní hlas → Nahrávací studio → Export pro trénování**)
se dotrénuje český hlas **Piper** z hlasu Jirka. Výsledek (`.onnx` a `.onnx.json`) pak v Překladači importujete
a hlas čte knihy i audioknihy **bez internetu**.

**Než začnete:** nahoře *Runtime / Běhové prostředí → Změnit typ běhového prostředí → GPU (T4)*.
Pak spouštějte buňky postupně (▶). Trénování trvá dlouho (u hodiny nahrávek řádově desítky hodin na bezplatné grafice T4) – bezplatný Colab
se po několika hodinách odpojí, průběh je ale na vašem Google Disku a buňkami 1–5 se pokračuje dál.
Rychlejší je Colab Pro nebo počítač s grafikou NVIDIA. Poslechnout si mezivýsledek jde kdykoli buňkou 6.

> Tento sešit používá oficiální trénování Piper (projekt *OHF-Voice/piper1-gpl*). Když se něco změní v jejich
> instalaci, chybu napíše buňka 2 – pošlete ji a sešit se upraví.

In [ ]:
# 1) Google Disk – sem se ukládá průběh trénování a hotový hlas
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/Prekladac-vlastni-hlas'
os.makedirs(WORK, exist_ok=True)
print('Pracovní složka:', WORK)

In [ ]:
# 2) Instalace trénování Piper (asi 5 minut)
# Oprava: espeakbridge (převod textu na hlásky) z oficiálního balíčku Piper
import glob, os, re, shutil, subprocess, sys, zipfile
SRC = '/content/piper1-gpl/src/piper'
if not glob.glob(SRC + '/espeakbridge*.so'):
    ver = re.search(r'version="([^"]+)"', open('/content/piper1-gpl/setup.py', encoding='utf-8').read())
    pkg = 'piper-tts==' + ver.group(1) if ver else 'piper-tts'
    print('Stahuji', pkg, '…')
    subprocess.run([sys.executable, '-m', 'pip', 'download', pkg, '--no-deps', '-q', '-d', '/content/wheel'], check=True)
    z = zipfile.ZipFile(glob.glob('/content/wheel/piper_tts-*.whl')[0])
    for n in z.namelist():
        if n.startswith('piper/espeakbridge') or n.startswith('piper/espeak-ng-data/'):
            z.extract(n, '/content/whl_x')
    shutil.copy(glob.glob('/content/whl_x/piper/espeakbridge*.so')[0], SRC)
    shutil.rmtree(SRC + '/espeak-ng-data', ignore_errors=True)
    shutil.copytree('/content/whl_x/piper/espeak-ng-data', SRC + '/espeak-ng-data')
sys.path.insert(0, '/content/piper1-gpl/src')
from piper.phonemize_espeak import EspeakPhonemizer
print('✓ Převod na hlásky funguje:', EspeakPhonemizer().phonemize('cs', 'Dobrý den.'))

✓ Převod na hlásky funguje: [['d', 'ˈ', 'o', 'b', 'r', 'i', 'ː', ' ', 'd', 'ˈ', 'e', 'n', '.']]


In [ ]:
# 3) ZIP(y) s nahrávkami z Překladače – nejspolehlivěji přes Google Disk:
#    drive.google.com → Můj disk → Prekladac-vlastni-hlas → přetáhnout ZIP (i několik najednou)
import glob, zipfile, os, shutil
zips = glob.glob(WORK + '/*.zip')
if not zips:
    try:
        from google.colab import files
        up = files.upload()  # okno pro nahrání (při blokovaných cookies nefunguje)
        for name, data in up.items():
            open(os.path.join(WORK, name), 'wb').write(data)
    except Exception as e:
        print('⚠ Okno pro nahrání se v prohlížeči nespustilo (' + type(e).__name__ + ').')
        print('  Nahrajte ZIP na https://drive.google.com do složky Prekladac-vlastni-hlas a spusťte buňku znovu.')
    zips = glob.glob(WORK + '/*.zip')
assert zips, 'Ve složce Prekladac-vlastni-hlas na Google Disku zatím není žádný ZIP.'

# všechny ZIPy se spojí do jednoho vstupu (stejné věty se nezdvojí)
shutil.rmtree('/content/data', ignore_errors=True)
shutil.rmtree('/content/cache', ignore_errors=True)  # nové nahrávky = nově připravit data pro trénink
DATA = '/content/data/hlas'
os.makedirs(DATA + '/wavs', exist_ok=True)
lines, seen, names = [], set(), []
for k, z in enumerate(sorted(zips, key=os.path.getmtime)):
    tmp = f'/content/zip{k}'
    shutil.rmtree(tmp, ignore_errors=True)
    zipfile.ZipFile(z).extractall(tmp)
    for meta in glob.glob(tmp + '/*/metadata.csv'):
        base = os.path.dirname(meta)
        names.append(os.path.basename(base))
        for line in open(meta, encoding='utf-8'):
            wav, _, text = line.rstrip('\n').partition('|')
            if not text or text.lower() in seen or not os.path.isfile(f'{base}/wavs/{wav}'):
                continue
            seen.add(text.lower())
            new = f'{len(lines) + 1:05d}.wav'
            shutil.copy(f'{base}/wavs/{wav}', f'{DATA}/wavs/{new}')
            lines.append(f'{new}|{text}')
open(DATA + '/metadata.csv', 'w', encoding='utf-8').write('\n'.join(lines) + '\n')
VOICE = names[-1] if names else 'muj_hlas'
print(f'ZIPů: {len(zips)} · hlas: {VOICE} · vět celkem: {len(lines)}')
if len(lines) < 300: print('⚠ Méně než 300 vět – hlas bude méně přesný; nahrajte víc, pokud to jde.')

ZIPů: 2 · hlas: muj_hlas · vět celkem: 309


In [ ]:
# 4) Výchozí hlas Jirka – stáhne se jen jednou a uloží na Google Disk
import os, glob, shutil
CKPT_START = WORK + '/jirka-start.ckpt'
PATH = 'cs/cs_CZ/jirka/medium/epoch=2099-step=92400.ckpt'
if os.path.isfile(CKPT_START):
    print('Jirka už je na Disku.')
else:
    local = glob.glob('/content/start/**/*.ckpt', recursive=True)
    if not local:
        from huggingface_hub import hf_hub_download
        for typ in ('model', 'dataset'):
            try:
                print(f'Stahuji Jirku (849 MB, typ {typ})…')
                local = [hf_hub_download('rhasspy/piper-checkpoints', PATH, repo_type=typ,
                                         local_dir='/content/start', token=False)]
                break
            except Exception as e:
                print('  nejde:', type(e).__name__, str(e)[:200])
    assert local, 'Jirku se nepodařilo stáhnout – pošlete mi celý tento výpis.'
    print('Staženo:', local[0], round(os.path.getsize(local[0]) / 1e6), 'MB')
    print('Kopíruji na Google Disk (může trvat několik minut)…')
    shutil.copy(local[0], CKPT_START + '.part')
    os.replace(CKPT_START + '.part', CKPT_START)
print('✓ Výchozí hlas:', CKPT_START, round(os.path.getsize(CKPT_START) / 1e6), 'MB')

Staženo: /content/start/cs/cs_CZ/jirka/medium/epoch=2099-step=92400.ckpt 849 MB
Kopíruji na Google Disk (může trvat několik minut)…
✓ Výchozí hlas: /content/drive/MyDrive/Prekladac-vlastni-hlas/jirka-start.ckpt 849 MB


In [ ]:
# 5) Trénování – průběh se ukládá na Disk; po odpojení stačí spustit znovu buňky 1–5 (pokračuje se od last.ckpt)
import glob
last = sorted(glob.glob(WORK + '/lightning_logs/*/checkpoints/last.ckpt'), key=os.path.getmtime)
# první spuštění: váhy z Jirky, vlastní běh od epochy 0 (warmstart); další spuštění: pokračování vlastního běhu
START = f'--ckpt_path "{last[-1]}"' if last else f'--model.warmstart_ckpt "{CKPT_START}"'
print('Pokračuji z', last[-1]) if last else print('Začínám z hlasu', CKPT_START)
EPOCHS = 2000   # doporučení projektu Piper pro dotrénování z méně než ~5 h nahrávek
!cd {WORK} && python3 -m piper.train fit \
  --data.voice_name "{VOICE}" \
  --data.csv_path "{DATA}/metadata.csv" \
  --data.audio_dir "{DATA}/wavs" \
  --model.sample_rate 22050 \
  --data.espeak_voice cs \
  --data.cache_dir /content/cache \
  --data.config_path "{WORK}/{VOICE}.onnx.json" \
  --data.batch_size 16 \
  --trainer.max_epochs {EPOCHS} \
  --trainer.default_root_dir "{WORK}" \
  {START}

Začínám z hlasu /content/drive/MyDrive/Prekladac-vlastni-hlas/jirka-start.ckpt
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/lightning/fabric/utilities/seed.py:44: No seed found, seed set to 0
Seed set to 0
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:piper.train.vits.dataset:Processing utterances...
INFO:piper.train.vi

In [ ]:
# 6) Export hlasu (.onnx + .onnx.json) a krátká ukázka
import glob, json
# nejnovější last.ckpt (projekt Piper doporučuje začít s ním; porovnat jde i val_mos / val_mel)
ck = sorted(glob.glob(WORK + '/lightning_logs/*/checkpoints/last.ckpt'), key=os.path.getmtime)[-1]
!python3 -m piper.train.export_onnx --checkpoint "{ck}" --output-file "{WORK}/{VOICE}.onnx"
cfg = json.load(open(f'{WORK}/{VOICE}.onnx.json', encoding='utf-8'))
cfg['dataset'] = VOICE
json.dump(cfg, open(f'{WORK}/{VOICE}.onnx.json', 'w', encoding='utf-8'), ensure_ascii=False)
!pip -q install piper-tts
!echo "Dobrý den, tohle je můj vlastní hlas v Překladači." | piper -m "{WORK}/{VOICE}.onnx" -f /content/ukazka.wav
from IPython.display import Audio
Audio('/content/ukazka.wav')

In [ ]:
# 7) Stažení – oba soubory dejte do jedné složky a v Překladači: Nastavení → Součásti → Importovat hlas (.onnx)
from google.colab import files
files.download(f'{WORK}/{VOICE}.onnx')
files.download(f'{WORK}/{VOICE}.onnx.json')